In [ ]:
import os
from dotenv import load_dotenv
from urllib.parse import quote_plus

import pandas as pd
import polars as pl
import duckdb
import pyodbc
from sqlalchemy import create_engine, URL

load_dotenv()   # looks for .env in the current directory and parent folders

# for k, v in os.environ.items():
#    print(f"{k} == {v}")

In [ ]:
PG = dict(host="localhost", port=5432, dbname="eBooks",
          user=os.environ["PGSQL_USER"], password=os.environ["PGSQL_PWD"])


MS_CONN_STR = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;DATABASE=IntroRDBMS;"      # named instance: SERVER=localhost\\SQLEXPRESS
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"          # needed for local servers with self-signed certs
)

In [ ]:

# PostgreSQL
pg_uri = (f"postgresql://{PG['user']}:{quote_plus(PG['password'])}"
          f"@{PG['host']}:{PG['port']}/{PG['dbname']}")

pg_engine = create_engine(URL.create(
    "postgresql+psycopg", username=PG["user"], password=PG["password"],
    host=PG["host"], port=PG["port"], database=PG["dbname"]))

# SQL Server (Windows auth)
ms_engine = create_engine(URL.create(
    "mssql+pyodbc", host="localhost", database="IntroRDBMS",
    query={"driver": "ODBC Driver 18 for SQL Server",
           "Trusted_Connection": "yes",
           "TrustServerCertificate": "yes"}))

In [ ]:
print(pg_uri)
print(pg_engine)
print(ms_engine)

In [ ]:
print(pd.read_sql("SELECT version()", pg_engine).iloc[0, 0])
print(pd.read_sql("SELECT @@VERSION AS v", ms_engine).iloc[0, 0])

In [ ]:
df_pd = pd.read_sql("SELECT * FROM public.genres", pg_engine)
df_pd.info()
df_pd.describe(include="all")

In [ ]:
pd.read_sql("""
    SELECT TABLE_SCHEMA, TABLE_NAME, TABLE_TYPE
    FROM   INFORMATION_SCHEMA.TABLES
    ORDER  BY 1, 2
""", ms_engine)

In [ ]:
# PostgreSQL via ConnectorX (fast, Arrow-based)
df_pl = pl.read_database_uri("SELECT * FROM public.books", pg_uri, engine="connectorx")

# SQL Server via pyodbc with Windows auth
with pyodbc.connect(MS_CONN_STR) as conn:
    df_ms_pl = pl.read_database("SELECT * FROM dbo.BookShop", connection=conn)

df_pl.schema
df_pl.describe()

In [ ]:
con = duckdb.connect()

In [ ]:
con.sql("INSTALL postgres; LOAD postgres;")

In [ ]:
con.sql(f"""
    ATTACH 'dbname={PG["dbname"]} user={PG["user"]} password={PG["password"]}
            host={PG["host"]} port={PG["port"]}' AS pg (TYPE postgres, READ_ONLY)
""")

In [ ]:
con.sql("SHOW ALL TABLES").show()
con.sql("DESCRIBE pg.public.genres").show()
con.sql("SELECT COUNT(*) FROM pg.public.genres").show()